# Chapter 14: Scoring

> En este capítulo se termina Alien Invasion: un botón Play, niveles que aumentan la velocidad y un sistema de puntaje.

## Summary

Se agrega un botón **Play** (Pygame no trae botones, así que se escribe una clase `Button` con un rectángulo y un texto): el juego empieza inactivo (`game_active = False`) y se inicia al hacer clic sobre el botón, detectado con `MOUSEBUTTONDOWN`, `pygame.mouse.get_pos()` y `rect.collidepoint()`. Al empezar se reinician las estadísticas y se oculta el cursor del mouse.
Para subir de nivel se dividen los settings en estáticos y dinámicos: `initialize_dynamic_settings()` los reinicia en cada partida e `increase_speed()` los multiplica por `speedup_scale` cada vez que se destruye una flota.
El puntaje lo muestra una clase `Scoreboard`: el texto se convierte en una imagen con `font.render()` y se dibuja con `blit()`. Se muestran el puntaje (redondeado a decenas y con comas), el puntaje máximo, el nivel y las naves restantes (dibujadas como un grupo de imágenes).
Cada alien vale más puntos en los niveles altos (`score_scale`), y `collisions.values()` permite sumar puntos por *todos* los aliens golpeados.

## Cheat Sheet

### Funciones, métodos y atributos

| Quiero... | Sintaxis | Ejemplo | Resultado / nota |
|---|---|---|---|
| Usar texto en Pygame | `import pygame.font` | | permite dibujar texto en la pantalla |
| Crear una fuente | `pygame.font.SysFont(None, tamaño)` | `pygame.font.SysFont(None, 48)` | `None` = fuente por defecto |
| Convertir texto en imagen | `font.render(texto, antialias, color_texto, color_fondo)` | `self.font.render(msg, True, self.text_color, self.button_color)` | `True` suaviza los bordes; sin color de fondo, el fondo es transparente |
| Rellenar solo una parte | `screen.fill(color, rect)` | `self.screen.fill(self.button_color, self.rect)` | dibuja un rectángulo de color |
| Dibujar una imagen | `screen.blit(imagen, rect)` | `self.screen.blit(self.msg_image, self.msg_image_rect)` | |
| Detectar un clic | `event.type == pygame.MOUSEBUTTONDOWN` | | se produce al hacer clic en cualquier parte |
| Posición del mouse | `pygame.mouse.get_pos()` | `mouse_pos = pygame.mouse.get_pos()` | tupla `(x, y)` |
| Saber si un punto está en un rect | `rect.collidepoint(punto)` | `self.play_button.rect.collidepoint(mouse_pos)` | `True` / `False` |
| Ocultar / mostrar el cursor | `pygame.mouse.set_visible(bool)` | `pygame.mouse.set_visible(False)` | `False` lo oculta |
| Redondear a decenas, centenas... | `round(número, -1)` | `round(1234, -1)` | `1230` (con `-2` redondea a centenas, etc.) |
| Poner comas a los miles | `f"{número:,}"` | `f"{1000000:,}"` | `'1,000,000'` (la parte `:,` es un *format specifier*) |
| Convertir a entero | `int(...)` | `int(self.alien_points * self.score_scale)` | sube los puntos en números enteros |
| Convertir a texto | `str(...)` | `str(self.stats.score)` | |
| Valores de un diccionario | `collisions.values()` | `for aliens in collisions.values():` | cada valor es la lista de aliens golpeados por una bala |
| Crear un grupo de naves | `Group()` | `from pygame.sprite import Group` | |
| Dibujar un grupo | `grupo.draw(superficie)` | `self.ships.draw(self.screen)` | |

### Clases y métodos nuevos

| Clase / método | Para qué sirve |
|---|---|
| `Button(ai_game, msg)` | crea un botón con un texto (`_prep_msg()`, `draw_button()`) |
| `AlienInvasion._check_play_button(mouse_pos)` | inicia una partida nueva al hacer clic en Play |
| `Settings.initialize_dynamic_settings()` | deja los settings que cambian durante el juego en sus valores iniciales |
| `Settings.increase_speed()` | aumenta las velocidades y los puntos por alien |
| `Scoreboard(ai_game)` | muestra puntaje, puntaje máximo, nivel y naves |
| `Scoreboard.prep_score()` / `prep_high_score()` / `prep_level()` / `prep_ships()` | convierten cada dato en una imagen (o un grupo de imágenes) |
| `Scoreboard.check_high_score()` | revisa si el puntaje actual superó al máximo |
| `Scoreboard.show_score()` | dibuja todo en la pantalla |

### Settings y estadísticas nuevos

| Nombre | Valor | Para qué sirve |
|---|---|---|
| `speedup_scale` | `1.1` | cuánto se acelera el juego en cada nivel |
| `score_scale` | `1.5` | cuánto aumenta el valor de cada alien en cada nivel |
| `alien_points` | `50` | puntos por alien (dinámico: se reinicia en cada partida) |
| `stats.score` | `0` | puntaje actual (se reinicia en `reset_stats()`) |
| `stats.high_score` | `0` | puntaje máximo (**nunca se reinicia**: se crea en `__init__()`) |
| `stats.level` | `1` | nivel actual (se reinicia en `reset_stats()`) |

### Problemas del capítulo

| Problema | Mensaje | Causa | Solución |
|---|---|---|---|
| El juego no se reinicia bien al hacer clic en Play por segunda vez | (no hay mensaje) | las condiciones del final de la partida no se reiniciaron | reiniciar estadísticas, vaciar los grupos, crear una flota nueva y centrar la nave |
| El área del botón sigue respondiendo en pleno juego | (no hay mensaje) | `_check_play_button()` no revisa si el juego está activo | usar `button_clicked and not self.game_active` |
| El cursor molesta durante el juego | (no hay mensaje) | sigue visible | `pygame.mouse.set_visible(False)` al empezar y `True` al terminar |
| Al empezar un juego se ve el puntaje anterior | (no hay mensaje) | el puntaje solo se vuelve a preparar cuando se golpea un alien | llamar a `prep_score()` (y `prep_level()`, `prep_ships()`) al pulsar Play |
| Una bala que golpea varios aliens solo da puntos por uno | (no hay mensaje) | se suma una sola vez por colisión | recorrer `collisions.values()` y multiplicar por `len(aliens)` |
| El juego se afecta al dejar un `print()` | (no hay mensaje) | quedó un `print()` de prueba | borrarlo después de verificar |

## Adding the Play Button

Ahora aparece un botón Play antes de empezar y cuando termina la partida, para volver a jugar.
Hoy el juego empieza apenas se ejecuta `alien_invasion.py`. Se cambia para que empiece **inactivo**:

In [ ]:
# alien_invasion.py
    def __init__(self):
        """Initialize the game, and create game resources."""
        pygame.init()
        # --snip--
        # Start Alien Invasion in an inactive state.
        self.game_active = False

### Creating a Button Class

Pygame no tiene una función para crear botones, así que se escribe una clase `Button` que crea un rectángulo relleno con un texto. Sirve para cualquier botón del juego. Primera parte, en `button.py`:

In [ ]:
# button.py
import pygame.font

class Button:
    """A class to build buttons for the game."""

    def __init__(self, ai_game, msg):
        """Initialize button attributes."""
        self.screen = ai_game.screen
        self.screen_rect = self.screen.get_rect()

        # Set the dimensions and properties of the button.
        self.width, self.height = 200, 50
        self.button_color = (0, 135, 0)
        self.text_color = (255, 255, 255)
        self.font = pygame.font.SysFont(None, 48)

        # Build the button's rect object and center it.
        self.rect = pygame.Rect(0, 0, self.width, self.height)
        self.rect.center = self.screen_rect.center

        # The button message needs to be prepped only once.
        self._prep_msg(msg)

- Se importa `pygame.font`, que permite dibujar texto en la pantalla.
- `__init__()` recibe `self`, el objeto `ai_game` y `msg`, el texto del botón.
- Se definen las dimensiones del botón (200 × 50), `button_color` (verde oscuro) y `text_color` (blanco).
- Para el texto se prepara un atributo `font`: `None` indica la fuente por defecto y `48` el tamaño.
- Se crea un rect para el botón y se centra igualando su `center` al de la pantalla.
- Pygame trabaja con texto **convirtiéndolo en una imagen**. De eso se encarga `_prep_msg()`:

In [ ]:
# button.py
    def _prep_msg(self, msg):
        """Turn msg into a rendered image and center text on the button."""
        self.msg_image = self.font.render(msg, True, self.text_color,
                self.button_color)
        self.msg_image_rect = self.msg_image.get_rect()
        self.msg_image_rect.center = self.rect.center

`font.render()` convierte el texto de `msg` en una imagen. Sus argumentos son el texto, un booleano para activar o no el **antialiasing** (suaviza los bordes del texto), el color del texto y el color de fondo. Aquí el fondo es el mismo color del botón (si no se pone un color de fondo, Pygame intenta renderizar el texto con fondo transparente).
Luego se crea un rect a partir de la imagen y se centra en el botón. Para mostrarlo en pantalla se crea `draw_button()`:

In [ ]:
# button.py
    def draw_button(self):
        """Draw blank button and then draw message."""
        self.screen.fill(self.button_color, self.rect)
        self.screen.blit(self.msg_image, self.msg_image_rect)

`screen.fill()` dibuja la parte rectangular del botón y `screen.blit()` dibuja la imagen del texto, pasándole la imagen y su rect. Con eso la clase `Button` está completa.

### Drawing the Button to the Screen

In [ ]:
# alien_invasion.py
# --snip--
from game_stats import GameStats
from button import Button

In [ ]:
# alien_invasion.py
    def __init__(self):
        # --snip--
        self.game_active = False

        # Make the Play button.
        self.play_button = Button(self, "Play")

Esto crea el botón con el texto `Play`, pero no lo dibuja. Para dibujarlo se llama a `draw_button()` en `_update_screen()`, **después** de los demás elementos y antes de `flip()`, para que quede por encima de todo:

In [ ]:
# alien_invasion.py
    def _update_screen(self):
        # --snip--
        self.aliens.draw(self.screen)

        # Draw the play button if the game is inactive.
        if not self.game_active:
            self.play_button.draw_button()

        pygame.display.flip()

El `if` hace que el botón solo aparezca cuando el juego está inactivo. Al ejecutar el juego se ve el botón Play en el centro de la pantalla.

### Starting the Game

Para iniciar un juego al hacer clic en Play se agrega un `elif` al final de `_check_events()` que vigila los eventos del mouse sobre el botón:

In [ ]:
# alien_invasion.py
    def _check_events(self):
        """Respond to keypresses and mouse events."""
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                # --snip--
            elif event.type == pygame.MOUSEBUTTONDOWN:
                mouse_pos = pygame.mouse.get_pos()
                self._check_play_button(mouse_pos)

Pygame detecta un evento `MOUSEBUTTONDOWN` cuando se hace clic **en cualquier parte** de la pantalla. Para responder solo a los clics sobre el botón se usa `pygame.mouse.get_pos()`, que devuelve una tupla con las coordenadas x e y del cursor al hacer clic, y se envían a `_check_play_button()`:

In [ ]:
# alien_invasion.py
    def _check_play_button(self, mouse_pos):
        """Start a new game when the player clicks Play."""
        if self.play_button.rect.collidepoint(mouse_pos):
            self.game_active = True

El método `collidepoint()` de un rect revisa si el punto del clic cae dentro del rect del botón. Si es así, `game_active` pasa a `True` y el juego empieza.

### Resetting the Game

El código anterior funciona la primera vez, pero no después de terminar una partida, porque no se reiniciaron las condiciones que la terminaron. Cada vez que el jugador presiona Play hay que reiniciar las estadísticas, vaciar los aliens y las balas, crear una flota nueva y centrar la nave:

In [ ]:
# alien_invasion.py
    def _check_play_button(self, mouse_pos):
        """Start a new game when the player clicks Play."""
        if self.play_button.rect.collidepoint(mouse_pos):
            # Reset the game statistics.
            self.stats.reset_stats()
            self.game_active = True

            # Get rid of any remaining bullets and aliens.
            self.bullets.empty()
            self.aliens.empty()

            # Create a new fleet and center the ship.
            self._create_fleet()
            self.ship.center_ship()

Al reiniciar las estadísticas el jugador recupera sus tres naves. Después se activa el juego, se vacían los grupos y se crea una flota nueva con la nave centrada.

### Deactivating the Play Button

El área del botón sigue respondiendo a los clics aunque no se vea, así que si se hace clic por accidente en medio de una partida, **el juego se reinicia**. Se arregla haciendo que el juego solo empiece cuando `game_active` es `False`:

In [ ]:
# alien_invasion.py
    def _check_play_button(self, mouse_pos):
        """Start a new game when the player clicks Play."""
        button_clicked = self.play_button.rect.collidepoint(mouse_pos)
        if button_clicked and not self.game_active:
            # Reset the game statistics.
            self.stats.reset_stats()
            # --snip--

`button_clicked` guarda un `True` o `False`, y el juego solo se reinicia si se hizo clic en Play **y** el juego no está activo.

### Hiding the Mouse Cursor

El cursor debe verse con el juego inactivo, pero molesta una vez que empieza. Se oculta al final del `if` de `_check_play_button()`:

In [ ]:
# alien_invasion.py
    def _check_play_button(self, mouse_pos):
        """Start a new game when the player clicks Play."""
        button_clicked = self.play_button.rect.collidepoint(mouse_pos)
        if button_clicked and not self.game_active:
            # --snip--
            # Hide the mouse cursor.
            pygame.mouse.set_visible(False)

Pasar `False` a `set_visible()` oculta el cursor cuando está sobre la ventana del juego. Y vuelve a aparecer cuando termina la partida, en `_ship_hit()`:

In [ ]:
# alien_invasion.py
    def _ship_hit(self):
        """Respond to ship being hit by alien."""
        if self.stats.ships_left > 0:
            # --snip--
        else:
            self.game_active = False
            pygame.mouse.set_visible(True)

Estos detalles hacen que el juego se vea más profesional y que el jugador se concentre en jugar y no en entender la interfaz.

### Comparison: Adding the Play Button

| | `_check_events()` | `_check_play_button()` |
|---|---|---|
| Qué detecta | que hubo un clic en **cualquier** parte (`MOUSEBUTTONDOWN`) | que el clic fue **sobre el botón** |
| Cómo | `pygame.mouse.get_pos()` | `rect.collidepoint(mouse_pos)` |
| Qué hace después | llama a `_check_play_button(mouse_pos)` | reinicia el juego y lo activa |

| | `if self.play_button.rect.collidepoint(mouse_pos):` | `if button_clicked and not self.game_active:` |
|---|---|---|
| Clic en el botón con el juego inactivo | empieza una partida | empieza una partida |
| Clic en el área del botón **durante** la partida | **reinicia** el juego por accidente | no pasa nada |

| | Juego inactivo | Juego activo |
|---|---|---|
| Botón Play | se dibuja | no se dibuja |
| Cursor del mouse | visible | oculto |
| `ship.update()`, balas y aliens | no se actualizan | se actualizan |

| | `Button.__init__()` | `Button._prep_msg()` | `Button.draw_button()` |
|---|---|---|---|
| Se ejecuta | una vez, al crear el botón | una vez, desde `__init__()` | en cada vuelta, si el juego está inactivo |
| Qué hace | define tamaño, colores, fuente y rect | convierte el texto en imagen y la centra | dibuja el rectángulo y el texto |

### Try it Yourself

> 14-1. Press P to Play:
Because Alien Invasion uses keyboard input to control the ship, it would be useful to start the game with a keypress. Add code that lets the player press P to start. It might help to move some code from _check_play_button() to a _start_game() method that can be called from _check_play_button() and _check_keydown_events().

> 14-2. Target Practice:
Create a rectangle at the right edge of the screen that moves up and down at a steady rate. Then on the left side of the screen, create a ship that the player can move up and down while firing bullets at the rectangular target. Add a Play button that starts the game, and when the player misses the target three times, end the game and make the Play button reappear. Let the player restart the game with this Play button.

In [ ]:
# 14-1


In [ ]:
# 14-2


## Leveling Up

Hoy, cuando el jugador destruye toda la flota "sube de nivel", pero la dificultad no cambia. Para hacerlo más desafiante, el juego se acelera cada vez que se limpia la pantalla.

### Modifying the Speed Settings

Primero se reorganiza `Settings` en dos grupos: los settings **estáticos** (no cambian) y los **dinámicos** (cambian durante el juego y deben reiniciarse con cada partida). Este es el `__init__()` de `settings.py`:

In [ ]:
# settings.py
    def __init__(self):
        """Initialize the game's static settings."""
        # Screen settings
        self.screen_width = 1200
        self.screen_height = 800
        self.bg_color = (230, 230, 230)

        # Ship settings
        self.ship_limit = 3

        # Bullet settings
        self.bullet_width = 3
        self.bullet_height = 15
        self.bullet_color = 60, 60, 60
        self.bullets_allowed = 3

        # Alien settings
        self.fleet_drop_speed = 10

        # How quickly the game speeds up
        self.speedup_scale = 1.1

        self.initialize_dynamic_settings()

Se agrega `speedup_scale`, que controla qué tan rápido se acelera el juego: un valor de `2` duplicaría la velocidad en cada nivel y un `1` la mantendría constante. Un valor como `1.1` aumenta la velocidad lo suficiente para ser desafiante, sin ser imposible.
Al final se llama a `initialize_dynamic_settings()`, que inicializa los valores que van a cambiar durante el juego:

In [ ]:
# settings.py
    def initialize_dynamic_settings(self):
        """Initialize settings that change throughout the game."""
        self.ship_speed = 1.5
        self.bullet_speed = 2.5
        self.alien_speed = 1.0

        # fleet_direction of 1 represents right; -1 represents left.
        self.fleet_direction = 1

Se incluye `fleet_direction` en este método para que los aliens siempre partan moviéndose a la derecha en una partida nueva. No hace falta aumentar `fleet_drop_speed`: si los aliens van más rápido por la pantalla, también bajan más rápido. Para acelerar el juego se escribe `increase_speed()`, que multiplica cada velocidad por `speedup_scale`:

In [ ]:
# settings.py
    def increase_speed(self):
        """Increase speed settings."""
        self.ship_speed *= self.speedup_scale
        self.bullet_speed *= self.speedup_scale
        self.alien_speed *= self.speedup_scale

Se llama a `increase_speed()` en `_check_bullet_alien_collisions()` cuando se destruye el último alien de la flota:

In [ ]:
# alien_invasion.py
    def _check_bullet_alien_collisions(self):
        # --snip--
        if not self.aliens:
            # Destroy existing bullets and create new fleet.
            self.bullets.empty()
            self._create_fleet()
            self.settings.increase_speed()

Con cambiar los valores de `ship_speed`, `alien_speed` y `bullet_speed` basta para acelerar todo el juego.

### Resetting the Speed

Cada vez que se empieza una partida hay que volver los settings a sus valores iniciales; si no, la nueva partida partiría con la velocidad de la anterior:

In [ ]:
# alien_invasion.py
    def _check_play_button(self, mouse_pos):
        """Start a new game when the player clicks Play."""
        button_clicked = self.play_button.rect.collidepoint(mouse_pos)
        if button_clicked and not self.game_active:
            # Reset the game settings.
            self.settings.initialize_dynamic_settings()
            # --snip--

Si el juego se vuelve demasiado difícil muy rápido, se baja `speedup_scale`; si no es lo bastante desafiante, se sube un poco. Hay que encontrar el punto justo: las primeras pantallas fáciles, las siguientes desafiantes pero posibles, y las posteriores casi imposibles.

### Comparison: Leveling Up

| | Settings estáticos | Settings dinámicos |
|---|---|---|
| Cambian durante el juego | No | Sí |
| Ejemplos | `screen_width`, `ship_limit`, `bullet_width`, `fleet_drop_speed`, `speedup_scale` | `ship_speed`, `bullet_speed`, `alien_speed`, `fleet_direction` |
| Dónde se definen | `__init__()` | `initialize_dynamic_settings()` |
| Cuándo se reinician | nunca | cada vez que se presiona Play |

| | `initialize_dynamic_settings()` | `increase_speed()` |
|---|---|---|
| Qué hace | pone las velocidades en sus valores iniciales | multiplica las velocidades por `speedup_scale` |
| Cuándo se llama | al crear `Settings` y al presionar Play | al destruir toda la flota |

| `speedup_scale` | Efecto |
|---|---|
| `1` | la velocidad no cambia |
| `1.1` | sube un 10 % por nivel (el valor del libro) |
| `2` | se duplica en cada nivel |

### Try it Yourself

> 14-3. Challenging Target Practice:
Start with your work from Exercise 14-2. Make the target move faster as the game progresses, and restart the target at the original speed when the player clicks Play.

> 14-4. Difficulty Levels:
Make a set of buttons for Alien Invasion that allows the player to select an appropriate starting difficulty level for the game. Each button should assign the appropriate values for the attributes in Settings needed to create different difficulty levels.

In [ ]:
# 14-3


In [ ]:
# 14-4


## Scoring

Se implementa un sistema de puntaje que lleva la cuenta del puntaje en tiempo real y muestra el puntaje máximo, el nivel y la cantidad de naves restantes.
El puntaje es una estadística del juego, así que se agrega un atributo `score` a `GameStats`. Se inicializa en `reset_stats()` (y no en `__init__()`) para reiniciarlo con cada partida:

In [ ]:
# game_stats.py
class GameStats:
    # --snip--
    def reset_stats(self):
        """Initialize statistics that can change during the game."""
        self.ships_left = self.settings.ship_limit
        self.score = 0

> En el libro este fragmento aparece escrito como `self.ai_settings.ship_limit`, pero el atributo se llama `self.settings` (así se definió en el Capítulo 13 y así está en el código final).

### Displaying the Score

Para mostrar el puntaje se crea una clase nueva, `Scoreboard`. Por ahora solo muestra el puntaje, pero después mostrará también el puntaje máximo, el nivel y las naves restantes. Primera parte, en `scoreboard.py`:

In [ ]:
# scoreboard.py
import pygame.font

class Scoreboard:
    """A class to report scoring information."""

    def __init__(self, ai_game):
        """Initialize scorekeeping attributes."""
        self.screen = ai_game.screen
        self.screen_rect = self.screen.get_rect()
        self.settings = ai_game.settings
        self.stats = ai_game.stats

        # Font settings for scoring information.
        self.text_color = (30, 30, 30)
        self.font = pygame.font.SysFont(None, 48)

        # Prepare the initial score image.
        self.prep_score()

Como `Scoreboard` escribe texto en la pantalla, empieza importando `pygame.font`. `__init__()` recibe `ai_game` para acceder a `settings`, `screen` y `stats`, que necesita para mostrar los valores que se registran. Luego se define el color del texto y se crea el objeto de fuente.
Para convertir el texto en imagen se llama a `prep_score()`:

In [ ]:
# scoreboard.py
    def prep_score(self):
        """Turn the score into a rendered image."""
        score_str = str(self.stats.score)
        self.score_image = self.font.render(score_str, True,
                self.text_color, self.settings.bg_color)

        # Display the score at the top right of the screen.
        self.score_rect = self.score_image.get_rect()
        self.score_rect.right = self.screen_rect.right - 20
        self.score_rect.top = 20

- Se convierte el valor numérico `stats.score` a texto con `str()` y se pasa a `render()`, que crea la imagen. Para que se lea bien, se pasan el color del texto y el color de fondo de la pantalla.
- El puntaje se ubica arriba a la derecha y crece hacia la izquierda a medida que el número se hace más ancho. Para que siempre quede alineado con el lado derecho, se crea `score_rect` con su borde derecho a 20 píxeles del borde derecho de la pantalla y su borde superior a 20 píxeles del borde superior.

Y `show_score()` dibuja la imagen:

In [ ]:
# scoreboard.py
    def show_score(self):
        """Draw score to the screen."""
        self.screen.blit(self.score_image, self.score_rect)

### Making a Scoreboard

Se crea una instancia de `Scoreboard` en `AlienInvasion`. Primero los `import`:

In [ ]:
# alien_invasion.py
# --snip--
from game_stats import GameStats
from scoreboard import Scoreboard
# --snip--

In [ ]:
# alien_invasion.py
    def __init__(self):
        # --snip--
        pygame.display.set_caption("Alien Invasion")

        # Create an instance to store game statistics,
        #   and create a scoreboard.
        self.stats = GameStats(self)
        self.sb = Scoreboard(self)
        # --snip--

Y se dibuja en `_update_screen()`, justo antes del botón Play:

In [ ]:
# alien_invasion.py
    def _update_screen(self):
        # --snip--
        self.aliens.draw(self.screen)

        # Draw the score information.
        self.sb.show_score()

        # Draw the play button if the game is inactive.
        # --snip--

Al ejecutar el juego aparece un `0` arriba a la derecha.

### Updating the Score as Aliens Are Shot Down

Para mostrar el puntaje en vivo, se actualiza `stats.score` cada vez que se golpea un alien y se llama a `prep_score()`. Primero, cuántos puntos vale cada alien, en `initialize_dynamic_settings()` (así se reinicia en cada partida):

In [ ]:
# settings.py
    def initialize_dynamic_settings(self):
        # --snip--
        # Scoring settings
        self.alien_points = 50

In [ ]:
# alien_invasion.py
    def _check_bullet_alien_collisions(self):
        """Respond to bullet-alien collisions."""
        # Remove any bullets and aliens that have collided.
        collisions = pygame.sprite.groupcollide(
                self.bullets, self.aliens, True, True)

        if collisions:
            self.stats.score += self.settings.alien_points
            self.sb.prep_score()
        # --snip--

Cuando una bala golpea un alien, Pygame devuelve el diccionario `collisions`. Si existe, se suma el valor del alien al puntaje y se llama a `prep_score()` para crear la imagen del puntaje actualizado.

### Resetting the Score

El puntaje solo se prepara después de golpear un alien, así que al empezar una partida nueva se seguiría viendo el puntaje anterior hasta golpear al primero. Se arregla preparando el puntaje al empezar:

In [ ]:
# alien_invasion.py
    def _check_play_button(self, mouse_pos):
        # --snip--
        if button_clicked and not self.game_active:
            # --snip--
            # Reset the game statistics.
            self.stats.reset_stats()
            self.sb.prep_score()
            # --snip--

### Making Sure to Score All Hits

Tal como está, el código podría no dar puntos por todos los aliens: si dos balas golpean aliens en la misma vuelta del loop, o si una bala extra ancha golpea varios, el jugador solo recibiría puntos por uno.
En `_check_bullet_alien_collisions()`, cada bala que choca con un alien es una llave del diccionario `collisions`, y su valor es una lista de los aliens con los que chocó. Se recorren los valores del diccionario para dar puntos por cada alien golpeado:

In [ ]:
# alien_invasion.py
    def _check_bullet_alien_collisions(self):
        # --snip--
        if collisions:
            for aliens in collisions.values():
                self.stats.score += self.settings.alien_points * len(aliens)
            self.sb.prep_score()
    # --snip--

Se multiplica el valor de un alien por la cantidad de aliens de cada lista y se suma al puntaje. Para probarlo, se puede cambiar el ancho de una bala a 300 píxeles y comprobar que se reciben puntos por cada alien golpeado, y luego devolverlo a su valor normal.

### Increasing Point Values

Como el juego se vuelve más difícil en cada nivel, los aliens de niveles posteriores deben valer más puntos. Se agrega un `score_scale` y se aumenta el valor del alien junto con la velocidad:

In [ ]:
# settings.py
class Settings:
    """A class to store all settings for Alien Invasion."""

    def __init__(self):
        # --snip--
        # How quickly the game speeds up
        self.speedup_scale = 1.1
        # How quickly the alien point values increase
        self.score_scale = 1.5

        self.initialize_dynamic_settings()

    def initialize_dynamic_settings(self):
        # --snip--
    def increase_speed(self):
        """Increase speed settings and alien point values."""
        self.ship_speed *= self.speedup_scale
        self.bullet_speed *= self.speedup_scale
        self.alien_speed *= self.speedup_scale

        self.alien_points = int(self.alien_points * self.score_scale)

Un pequeño aumento de velocidad (`1.1`) hace el juego más difícil rápidamente, pero para notar la diferencia en el puntaje se necesita un cambio mayor (`1.5`). Se usa `int()` para que los puntos aumenten en números enteros.
Para ver el valor de cada alien, se puede agregar un `print(self.alien_points)` temporal en `increase_speed()`. **Hay que borrarlo después de verificar**, porque afecta el rendimiento y distrae al jugador.

### Rounding the Score

Los juegos de arcade suelen mostrar el puntaje en múltiplos de 10, y con comas en los números grandes. Se cambia en `Scoreboard`:

In [ ]:
# scoreboard.py
    def prep_score(self):
        """Turn the score into a rendered image."""
        rounded_score = round(self.stats.score, -1)
        score_str = f"{rounded_score:,}"
        self.score_image = self.font.render(score_str, True,
                self.text_color, self.settings.bg_color)
        # --snip--

`round()` normalmente redondea un decimal a la cantidad de decimales que indica el segundo argumento. Pero si el segundo argumento es **negativo**, redondea al múltiplo de 10, 100, 1.000, etc. más cercano: con `-1` redondea a la decena.
En el f-string se usa un **format specifier**: una secuencia de caracteres que modifica cómo se presenta el valor de una variable. La secuencia `:,` le dice a Python que inserte comas en los lugares correctos: `1,000,000` en vez de `1000000`.

### High Scores

Los jugadores quieren superar el puntaje máximo, así que se guarda y se muestra. El puntaje máximo se guarda en `GameStats`, y se inicializa en `__init__()` (no en `reset_stats()`) porque **nunca se reinicia**:

In [ ]:
# game_stats.py
    def __init__(self, ai_game):
        # --snip--
        # High score should never be reset.
        self.high_score = 0

Luego, en `Scoreboard`, se prepara una nueva imagen para el puntaje máximo:

In [ ]:
# scoreboard.py
    def __init__(self, ai_game):
        # --snip--
        # Prepare the initial score images.
        self.prep_score()
        self.prep_high_score()

In [ ]:
# scoreboard.py
    def prep_high_score(self):
        """Turn the high score into a rendered image."""
        high_score = round(self.stats.high_score, -1)
        high_score_str = f"{high_score:,}"
        self.high_score_image = self.font.render(high_score_str, True,
                self.text_color, self.settings.bg_color)

        # Center the high score at the top of the screen.
        self.high_score_rect = self.high_score_image.get_rect()
        self.high_score_rect.centerx = self.screen_rect.centerx
        self.high_score_rect.top = self.score_rect.top

El puntaje máximo se redondea a decenas, se le ponen comas, se convierte en imagen, se centra horizontalmente y su borde superior se iguala al de la imagen del puntaje. `show_score()` ahora dibuja el puntaje arriba a la derecha y el puntaje máximo arriba al centro:

In [ ]:
# scoreboard.py
    def show_score(self):
        """Draw score to the screen."""
        self.screen.blit(self.score_image, self.score_rect)
        self.screen.blit(self.high_score_image, self.high_score_rect)

Para revisar si hay un nuevo récord se escribe `check_high_score()`:

In [ ]:
# scoreboard.py
    def check_high_score(self):
        """Check to see if there's a new high score."""
        if self.stats.score > self.stats.high_score:
            self.stats.high_score = self.stats.score
            self.prep_high_score()

Si el puntaje actual es mayor que el máximo, se actualiza el máximo y se llama a `prep_high_score()` para actualizar su imagen. Se llama a `check_high_score()` cada vez que se golpea un alien, **después** de actualizar el puntaje:

In [ ]:
# alien_invasion.py
    def _check_bullet_alien_collisions(self):
        # --snip--
        if collisions:
            for aliens in collisions.values():
                self.stats.score += self.settings.alien_points * len(aliens)
            self.sb.prep_score()
            self.sb.check_high_score()
        # --snip--

La primera vez que se juega, el puntaje es el máximo y aparece dos veces. En la segunda partida, el máximo aparece al centro y el puntaje actual a la derecha.

### Displaying the Level

Primero, un atributo `level` en `GameStats`, que se inicializa en `reset_stats()` para reiniciarlo en cada partida:

In [ ]:
# game_stats.py
    def reset_stats(self):
        """Initialize statistics that can change during the game."""
        self.ships_left = self.settings.ship_limit
        self.score = 0
        self.level = 1

Y en `Scoreboard`, `__init__()` llama a un nuevo método, `prep_level()`:

In [ ]:
# scoreboard.py
    def __init__(self, ai_game):
        # --snip--
        self.prep_high_score()
        self.prep_level()

In [ ]:
# scoreboard.py
    def prep_level(self):
        """Turn the level into a rendered image."""
        level_str = str(self.stats.level)
        self.level_image = self.font.render(level_str, True,
                self.text_color, self.settings.bg_color)

        # Position the level below the score.
        self.level_rect = self.level_image.get_rect()
        self.level_rect.right = self.score_rect.right
        self.level_rect.top = self.score_rect.bottom + 10

`prep_level()` crea una imagen con el valor de `stats.level`, iguala su `right` al del puntaje y pone su borde superior 10 píxeles debajo del puntaje, para dejar un espacio entre ambos. También se actualiza `show_score()`:

In [ ]:
# scoreboard.py
    def show_score(self):
        """Draw scores and level to the screen."""
        self.screen.blit(self.score_image, self.score_rect)
        self.screen.blit(self.high_score_image, self.high_score_rect)
        self.screen.blit(self.level_image, self.level_rect)

El nivel sube y su imagen se actualiza en `_check_bullet_alien_collisions()` cuando se destruye una flota:

In [ ]:
# alien_invasion.py
    def _check_bullet_alien_collisions(self):
        # --snip--
        if not self.aliens:
            # Destroy existing bullets and create new fleet.
            self.bullets.empty()
            self._create_fleet()
            self.settings.increase_speed()

            # Increase level.
            self.stats.level += 1
            self.sb.prep_level()

Y para que la imagen del nivel se actualice al empezar una partida nueva, también se llama a `prep_level()` al presionar Play, justo después de `prep_score()`:

In [ ]:
# alien_invasion.py
    def _check_play_button(self, mouse_pos):
        # --snip--
        if button_clicked and not self.game_active:
            # --snip--
            self.sb.prep_score()
            self.sb.prep_level()
            # --snip--

> En algunos juegos clásicos los puntajes llevan etiquetas, como Score, High Score y Level. Aquí se omiten, porque el significado de cada número se entiende al jugar. Para agregarlas, se ponen en los strings justo antes de las llamadas a `font.render()` en `Scoreboard`.

### Displaying the Number of Ships

Por último se muestran las naves restantes, pero con una imagen: se dibujan naves arriba a la izquierda, como en muchos juegos de arcade. Para poder crear un grupo de naves, `Ship` debe heredar de `Sprite`:

In [ ]:
# ship.py
import pygame
from pygame.sprite import Sprite

class Ship(Sprite):
    """A class to manage the ship."""

    def __init__(self, ai_game):
        """Initialize the ship and set its starting position."""
        super().__init__()
        # --snip--

En `Scoreboard` se importan `Group` y `Ship`, y `__init__()` guarda la instancia del juego (la necesita para crear naves) y llama a `prep_ships()` después de `prep_level()`:

In [ ]:
# scoreboard.py
import pygame.font
from pygame.sprite import Group

from ship import Ship

In [ ]:
# scoreboard.py
    def __init__(self, ai_game):
        """Initialize scorekeeping attributes."""
        self.ai_game = ai_game
        self.screen = ai_game.screen
        # --snip--
        self.prep_level()
        self.prep_ships()

In [ ]:
# scoreboard.py
    def prep_ships(self):
        """Show how many ships are left."""
        self.ships = Group()
        for ship_number in range(self.stats.ships_left):
            ship = Ship(self.ai_game)
            ship.rect.x = 10 + ship_number * ship.rect.width
            ship.rect.y = 10
            self.ships.add(ship)

- Se crea un grupo vacío, `self.ships`.
- Un loop corre una vez por cada nave que le queda al jugador. En cada vuelta se crea una nave y se ubica su x para que queden una al lado de la otra, con un margen de 10 píxeles a la izquierda del grupo. La y queda a 10 píxeles del borde superior, para que las naves aparezcan arriba a la izquierda.
- Cada nave se agrega al grupo.

Y se dibujan con `draw()` del grupo en `show_score()`:

In [ ]:
# scoreboard.py
    def show_score(self):
        """Draw scores, level, and ships to the screen."""
        self.screen.blit(self.score_image, self.score_rect)
        self.screen.blit(self.high_score_image, self.high_score_rect)
        self.screen.blit(self.level_image, self.level_rect)
        self.ships.draw(self.screen)

Para mostrar cuántas naves tiene el jugador al empezar, se llama a `prep_ships()` en `_check_play_button()`:

In [ ]:
# alien_invasion.py
    def _check_play_button(self, mouse_pos):
        # --snip--
        if button_clicked and not self.game_active:
            # --snip--
            self.sb.prep_level()
            self.sb.prep_ships()
            # --snip--

Y también cuando se golpea una nave, para actualizar las imágenes. Se llama después de restar a `ships_left`:

In [ ]:
# alien_invasion.py
    def _ship_hit(self):
        """Respond to ship being hit by alien."""
        if self.stats.ships_left > 0:
            # Decrement ships_left, and update scoreboard.
            self.stats.ships_left -= 1
            self.sb.prep_ships()
            # --snip--

### Comparison: Scoring

| | `stats.score` | `stats.high_score` | `stats.level` | `stats.ships_left` |
|---|---|---|---|---|
| Se define en | `reset_stats()` | `__init__()` | `reset_stats()` | `reset_stats()` |
| Se reinicia con cada partida | Sí | **No** | Sí | Sí |
| Valor inicial | `0` | `0` | `1` | `settings.ship_limit` |
| Se muestra con | `prep_score()` | `prep_high_score()` | `prep_level()` | `prep_ships()` |
| Dónde se dibuja | arriba a la derecha | arriba al centro | debajo del puntaje | arriba a la izquierda |

| | `prep_*()` | `show_score()` |
|---|---|---|
| Qué hace | convierte un dato en una imagen (o un grupo de imágenes) y calcula su posición | dibuja todas las imágenes en la pantalla |
| Cuándo se llama | solo cuando el dato cambia (al golpear un alien, subir de nivel, perder una nave o empezar una partida) | en cada vuelta, desde `_update_screen()` |

| | Un `if collisions:` simple | Con `for aliens in collisions.values():` |
|---|---|---|
| Puntos por alien | `alien_points` (una sola vez por colisión) | `alien_points * len(aliens)` (por cada alien golpeado) |
| Si una bala golpea varios aliens | da puntos por uno | da puntos por todos |

Con `stats.score = 1234`:

| Expresión | Resultado |
|---|---|
| `round(1234, -1)` | `1230` |
| `round(1234, -2)` | `1200` |
| `f"{1234:,}"` | `'1,234'` |
| `f"{round(1234567, -1):,}"` | `'1,234,570'` |

| | Nave del juego | Naves de `Scoreboard` |
|---|---|---|
| Cuántas hay | una | `ships_left`, en un `Group` |
| Para qué sirven | jugar | solo mostrar cuántas naves quedan |
| Posición | abajo al centro, móvil | arriba a la izquierda, fija (`10 + ship_number * ship.rect.width`) |

### Try it Yourself

> 14-5. All-Time High Score:
The high score is reset every time a player closes and restarts Alien Invasion. Fix this by writing the high score to a file before calling sys.exit() and reading in the high score when initializing its value in GameStats.

> 14-6. Refactoring:
Look for methods that are doing more than one task, and refactor them to organize your code and make it efficient. For example, move some of the code in _check_bullet_alien_collisions(), which starts a new level when the fleet of aliens has been destroyed, to a function called start_new_level(). Also, move the four separate method calls in the __init__() method in Scoreboard to a method called prep_images() to shorten __init__(). The prep_images() method could also help simplify _check_play_button() or start_game() if you’ve already refactored _check_play_button().

> 14-7. Expanding the Game:
Think of a way to expand Alien Invasion. For example, you could program the aliens to shoot bullets down at your ship. You can also add shields for your ship to hide behind, which can be destroyed by bullets from either side. Or you can use something like the pygame.mixer module to add sound effects, such as explosions and shooting sounds.

> 14-8. Sideways Shooter, Final Version:
Continue developing Sideways Shooter, using everything we’ve done in this project. Add a Play button, make the game speed up at appropriate points, and develop a scoring system. Be sure to refactor your code as you work, and look for opportunities to customize the game beyond what has been shown in this chapter.

In [ ]:
# 14-5


In [ ]:
# 14-6


In [ ]:
# 14-7


In [ ]:
# 14-8


## Final Code

Los archivos del proyecto terminado (se ejecuta solo `alien_invasion.py`; las imágenes `ship.bmp` y `alien.bmp` van en la carpeta `images`).

**alien_invasion.py**

In [ ]:
# alien_invasion.py
import sys
from time import sleep

import pygame

from settings import Settings
from game_stats import GameStats
from scoreboard import Scoreboard
from button import Button
from ship import Ship
from bullet import Bullet
from alien import Alien


class AlienInvasion:
    """Overall class to manage game assets and behavior."""

    def __init__(self):
        """Initialize the game, and create game resources."""
        pygame.init()
        self.clock = pygame.time.Clock()
        self.settings = Settings()

        self.screen = pygame.display.set_mode(
            (self.settings.screen_width, self.settings.screen_height))
        pygame.display.set_caption("Alien Invasion")

        # Create an instance to store game statistics,
        #   and create a scoreboard.
        self.stats = GameStats(self)
        self.sb = Scoreboard(self)

        self.ship = Ship(self)
        self.bullets = pygame.sprite.Group()
        self.aliens = pygame.sprite.Group()

        self._create_fleet()

        # Start Alien Invasion in an inactive state.
        self.game_active = False

        # Make the Play button.
        self.play_button = Button(self, "Play")

    def run_game(self):
        """Start the main loop for the game."""
        while True:
            self._check_events()

            if self.game_active:
                self.ship.update()
                self._update_bullets()
                self._update_aliens()

            self._update_screen()
            self.clock.tick(60)

    def _check_events(self):
        """Respond to keypresses and mouse events."""
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                sys.exit()
            elif event.type == pygame.KEYDOWN:
                self._check_keydown_events(event)
            elif event.type == pygame.KEYUP:
                self._check_keyup_events(event)
            elif event.type == pygame.MOUSEBUTTONDOWN:
                mouse_pos = pygame.mouse.get_pos()
                self._check_play_button(mouse_pos)

    def _check_play_button(self, mouse_pos):
        """Start a new game when the player clicks Play."""
        button_clicked = self.play_button.rect.collidepoint(mouse_pos)
        if button_clicked and not self.game_active:
            # Reset the game settings.
            self.settings.initialize_dynamic_settings()

            # Reset the game statistics.
            self.stats.reset_stats()
            self.sb.prep_score()
            self.sb.prep_level()
            self.sb.prep_ships()
            self.game_active = True

            # Get rid of any remaining bullets and aliens.
            self.bullets.empty()
            self.aliens.empty()

            # Create a new fleet and center the ship.
            self._create_fleet()
            self.ship.center_ship()

            # Hide the mouse cursor.
            pygame.mouse.set_visible(False)

    def _check_keydown_events(self, event):
        """Respond to keypresses."""
        if event.key == pygame.K_RIGHT:
            self.ship.moving_right = True
        elif event.key == pygame.K_LEFT:
            self.ship.moving_left = True
        elif event.key == pygame.K_q:
            sys.exit()
        elif event.key == pygame.K_SPACE:
            self._fire_bullet()

    def _check_keyup_events(self, event):
        """Respond to key releases."""
        if event.key == pygame.K_RIGHT:
            self.ship.moving_right = False
        elif event.key == pygame.K_LEFT:
            self.ship.moving_left = False

    def _fire_bullet(self):
        """Create a new bullet and add it to the bullets group."""
        if len(self.bullets) < self.settings.bullets_allowed:
            new_bullet = Bullet(self)
            self.bullets.add(new_bullet)

    def _update_bullets(self):
        """Update position of bullets and get rid of old bullets."""
        # Update bullet positions.
        self.bullets.update()

        # Get rid of bullets that have disappeared.
        for bullet in self.bullets.copy():
            if bullet.rect.bottom <= 0:
                self.bullets.remove(bullet)

        self._check_bullet_alien_collisions()

    def _check_bullet_alien_collisions(self):
        """Respond to bullet-alien collisions."""
        # Remove any bullets and aliens that have collided.
        collisions = pygame.sprite.groupcollide(
                self.bullets, self.aliens, True, True)

        if collisions:
            for aliens in collisions.values():
                self.stats.score += self.settings.alien_points * len(aliens)
            self.sb.prep_score()
            self.sb.check_high_score()

        if not self.aliens:
            # Destroy existing bullets and create new fleet.
            self.bullets.empty()
            self._create_fleet()
            self.settings.increase_speed()

            # Increase level.
            self.stats.level += 1
            self.sb.prep_level()

    def _ship_hit(self):
        """Respond to the ship being hit by an alien."""
        if self.stats.ships_left > 0:
            # Decrement ships_left, and update scoreboard.
            self.stats.ships_left -= 1
            self.sb.prep_ships()

            # Get rid of any remaining bullets and aliens.
            self.bullets.empty()
            self.aliens.empty()

            # Create a new fleet and center the ship.
            self._create_fleet()
            self.ship.center_ship()

            # Pause.
            sleep(0.5)
        else:
            self.game_active = False
            pygame.mouse.set_visible(True)

    def _update_aliens(self):
        """Check if the fleet is at an edge, then update positions."""
        self._check_fleet_edges()
        self.aliens.update()

        # Look for alien-ship collisions.
        if pygame.sprite.spritecollideany(self.ship, self.aliens):
            self._ship_hit()

        # Look for aliens hitting the bottom of the screen.
        self._check_aliens_bottom()

    def _check_aliens_bottom(self):
        """Check if any aliens have reached the bottom of the screen."""
        for alien in self.aliens.sprites():
            if alien.rect.bottom >= self.settings.screen_height:
                # Treat this the same as if the ship got hit.
                self._ship_hit()
                break

    def _create_fleet(self):
        """Create the fleet of aliens."""
        # Create an alien and keep adding aliens until there's no room left.
        # Spacing between aliens is one alien width and one alien height.
        alien = Alien(self)
        alien_width, alien_height = alien.rect.size

        current_x, current_y = alien_width, alien_height
        while current_y < (self.settings.screen_height - 3 * alien_height):
            while current_x < (self.settings.screen_width - 2 * alien_width):
                self._create_alien(current_x, current_y)
                current_x += 2 * alien_width

            # Finished a row; reset x value, and increment y value.
            current_x = alien_width
            current_y += 2 * alien_height

    def _create_alien(self, x_position, y_position):
        """Create an alien and place it in the fleet."""
        new_alien = Alien(self)
        new_alien.x = x_position
        new_alien.rect.x = x_position
        new_alien.rect.y = y_position
        self.aliens.add(new_alien)

    def _check_fleet_edges(self):
        """Respond appropriately if any aliens have reached an edge."""
        for alien in self.aliens.sprites():
            if alien.check_edges():
                self._change_fleet_direction()
                break

    def _change_fleet_direction(self):
        """Drop the entire fleet and change the fleet's direction."""
        for alien in self.aliens.sprites():
            alien.rect.y += self.settings.fleet_drop_speed
        self.settings.fleet_direction *= -1

    def _update_screen(self):
        """Update images on the screen, and flip to the new screen."""
        self.screen.fill(self.settings.bg_color)
        for bullet in self.bullets.sprites():
            bullet.draw_bullet()
        self.ship.blitme()
        self.aliens.draw(self.screen)

        # Draw the score information.
        self.sb.show_score()

        # Draw the play button if the game is inactive.
        if not self.game_active:
            self.play_button.draw_button()

        pygame.display.flip()


if __name__ == '__main__':
    # Make a game instance, and run the game.
    ai = AlienInvasion()
    ai.run_game()

**alien.py**

In [ ]:
# alien.py
import pygame

from pygame.sprite import Sprite


class Alien(Sprite):
    """A class to represent a single alien in the fleet."""

    def __init__(self, ai_game):
        """Initialize the alien and set its starting position."""
        super().__init__()
        self.screen = ai_game.screen
        self.settings = ai_game.settings

        # Load the alien image and set its rect attribute.
        self.image = pygame.image.load('images/alien.bmp')
        self.rect = self.image.get_rect()

        # Start each new alien near the top left of the screen.
        self.rect.x = self.rect.width
        self.rect.y = self.rect.height

        # Store the alien's exact horizontal position.
        self.x = float(self.rect.x)

    def check_edges(self):
        """Return True if alien is at edge of screen."""
        screen_rect = self.screen.get_rect()
        return (self.rect.right >= screen_rect.right) or (self.rect.left <= 0)

    def update(self):
        """Move the alien right or left."""
        self.x += self.settings.alien_speed * self.settings.fleet_direction
        self.rect.x = self.x

**bullet.py**

In [ ]:
# bullet.py
import pygame
from pygame.sprite import Sprite

class Bullet(Sprite):
    """A class to manage bullets fired from the ship."""

    def __init__(self, ai_game):
        """Create a bullet object at the ship's current position."""
        super().__init__()
        self.screen = ai_game.screen
        self.settings = ai_game.settings
        self.color = self.settings.bullet_color

        # Create a bullet rect at (0, 0) and then set correct position.
        self.rect = pygame.Rect(0, 0, self.settings.bullet_width,
            self.settings.bullet_height)
        self.rect.midtop = ai_game.ship.rect.midtop

        # Store the bullet's position as a float.
        self.y = float(self.rect.y)

    def update(self):
        """Move the bullet up the screen."""
        # Update the exact position of the bullet.
        self.y -= self.settings.bullet_speed
        # Update the rect position.
        self.rect.y = self.y

    def draw_bullet(self):
        """Draw the bullet to the screen."""
        pygame.draw.rect(self.screen, self.color, self.rect)

**button.py**

In [ ]:
# button.py
import pygame.font


class Button:
    """A class to build buttons for the game."""

    def __init__(self, ai_game, msg):
        """Initialize button attributes."""
        self.screen = ai_game.screen
        self.screen_rect = self.screen.get_rect()

        # Set the dimensions and properties of the button.
        self.width, self.height = 200, 50
        self.button_color = (0, 135, 0)
        self.text_color = (255, 255, 255)
        self.font = pygame.font.SysFont(None, 48)

        # Build the button's rect object and center it.
        self.rect = pygame.Rect(0, 0, self.width, self.height)
        self.rect.center = self.screen_rect.center

        # The button message needs to be prepped only once.
        self._prep_msg(msg)

    def _prep_msg(self, msg):
        """Turn msg into a rendered image and center text on the button."""
        self.msg_image = self.font.render(msg, True, self.text_color,
                self.button_color)
        self.msg_image_rect = self.msg_image.get_rect()
        self.msg_image_rect.center = self.rect.center

    def draw_button(self):
        """Draw blank button and then draw message."""
        self.screen.fill(self.button_color, self.rect)
        self.screen.blit(self.msg_image, self.msg_image_rect)

**game_stats.py**

In [ ]:
# game_stats.py
class GameStats:
    """Track statistics for Alien Invasion."""

    def __init__(self, ai_game):
        """Initialize statistics."""
        self.settings = ai_game.settings
        self.reset_stats()

        # High score should never be reset.
        self.high_score = 0

    def reset_stats(self):
        """Initialize statistics that can change during the game."""
        self.ships_left = self.settings.ship_limit
        self.score = 0
        self.level = 1

**scoreboard.py**

In [ ]:
# scoreboard.py
import pygame.font
from pygame.sprite import Group

from ship import Ship


class Scoreboard:
    """A class to report scoring information."""

    def __init__(self, ai_game):
        """Initialize scorekeeping attributes."""
        self.ai_game = ai_game
        self.screen = ai_game.screen
        self.screen_rect = self.screen.get_rect()
        self.settings = ai_game.settings
        self.stats = ai_game.stats

        # Font settings for scoring information.
        self.text_color = (30, 30, 30)
        self.font = pygame.font.SysFont(None, 48)

        # Prepare the initial score images.
        self.prep_score()
        self.prep_high_score()
        self.prep_level()
        self.prep_ships()

    def prep_score(self):
        """Turn the score into a rendered image."""
        rounded_score = round(self.stats.score, -1)
        score_str = f"{rounded_score:,}"
        self.score_image = self.font.render(score_str, True,
                self.text_color, self.settings.bg_color)

        # Display the score at the top right of the screen.
        self.score_rect = self.score_image.get_rect()
        self.score_rect.right = self.screen_rect.right - 20
        self.score_rect.top = 20

    def prep_high_score(self):
        """Turn the high score into a rendered image."""
        high_score = round(self.stats.high_score, -1)
        high_score_str = f"{high_score:,}"
        self.high_score_image = self.font.render(high_score_str, True,
                self.text_color, self.settings.bg_color)
        
        # Center the high score at the top of the screen.
        self.high_score_rect = self.high_score_image.get_rect()
        self.high_score_rect.centerx = self.screen_rect.centerx
        self.high_score_rect.top = self.score_rect.top

    def prep_level(self):
        """Turn the level into a rendered image."""
        level_str = str(self.stats.level)
        self.level_image = self.font.render(level_str, True,
                self.text_color, self.settings.bg_color)

        # Position the level below the score.
        self.level_rect = self.level_image.get_rect()
        self.level_rect.right = self.score_rect.right
        self.level_rect.top = self.score_rect.bottom + 10

    def prep_ships(self):
        """Show how many ships are left."""
        self.ships = Group()
        for ship_number in range(self.stats.ships_left):
            ship = Ship(self.ai_game)
            ship.rect.x = 10 + ship_number * ship.rect.width
            ship.rect.y = 10
            self.ships.add(ship)

    def check_high_score(self):
        """Check to see if there's a new high score."""
        if self.stats.score > self.stats.high_score:
            self.stats.high_score = self.stats.score
            self.prep_high_score()

    def show_score(self):
        """Draw scores, level, and ships to the screen."""
        self.screen.blit(self.score_image, self.score_rect)
        self.screen.blit(self.high_score_image, self.high_score_rect)
        self.screen.blit(self.level_image, self.level_rect)
        self.ships.draw(self.screen)

**settings.py**

In [ ]:
# settings.py
class Settings:
    """A class to store all settings for Alien Invasion."""

    def __init__(self):
        """Initialize the game's static settings."""
        # Screen settings
        self.screen_width = 1200
        self.screen_height = 800
        self.bg_color = (230, 230, 230)

        # Ship settings
        self.ship_limit = 3

        # Bullet settings
        self.bullet_width = 3
        self.bullet_height = 15
        self.bullet_color = (60, 60, 60)
        self.bullets_allowed = 3

        # Alien settings
        self.fleet_drop_speed = 10

        # How quickly the game speeds up
        self.speedup_scale = 1.1
        # How quickly the alien point values increase
        self.score_scale = 1.5

        self.initialize_dynamic_settings()

    def initialize_dynamic_settings(self):
        """Initialize settings that change throughout the game."""
        self.ship_speed = 1.5
        self.bullet_speed = 2.5
        self.alien_speed = 1.0

        # fleet_direction of 1 represents right; -1 represents left.
        self.fleet_direction = 1

        # Scoring settings
        self.alien_points = 50

    def increase_speed(self):
        """Increase speed settings and alien point values."""
        self.ship_speed *= self.speedup_scale
        self.bullet_speed *= self.speedup_scale
        self.alien_speed *= self.speedup_scale

        self.alien_points = int(self.alien_points * self.score_scale)

**ship.py**

In [ ]:
# ship.py
import pygame
from pygame.sprite import Sprite


class Ship(Sprite):
    """A class to manage the ship."""

    def __init__(self, ai_game):
        """Initialize the ship and set its starting position."""
        super().__init__()
        self.screen = ai_game.screen
        self.settings = ai_game.settings
        self.screen_rect = ai_game.screen.get_rect()

        # Load the ship image and get its rect.
        self.image = pygame.image.load('images/ship.bmp')
        self.rect = self.image.get_rect()

        # Start each new ship at the bottom center of the screen.
        self.rect.midbottom = self.screen_rect.midbottom

        # Store a float for the ship's exact horizontal position.
        self.x = float(self.rect.x)

        # Movement flags; start with a ship that's not moving.
        self.moving_right = False
        self.moving_left = False

    def center_ship(self):
        """Center the ship on the screen."""
        self.rect.midbottom = self.screen_rect.midbottom
        self.x = float(self.rect.x)

    def update(self):
        """Update the ship's position based on movement flags."""
        # Update the ship's x value, not the rect.
        if self.moving_right and self.rect.right < self.screen_rect.right:
            self.x += self.settings.ship_speed
        if self.moving_left and self.rect.left > 0:
            self.x -= self.settings.ship_speed
            
        # Update rect object from self.x.
        self.rect.x = self.x

    def blitme(self):
        """Draw the ship at its current location."""
        self.screen.blit(self.image, self.rect)

## Comparison: Chapter 14

| Si necesito... | Uso |
|---|---|
| Un botón en Pygame | clase `Button`: un `Rect` + texto convertido en imagen |
| Texto en la pantalla | `pygame.font.SysFont()` + `font.render()` + `screen.blit()` |
| Detectar un clic sobre un elemento | `MOUSEBUTTONDOWN` + `pygame.mouse.get_pos()` + `rect.collidepoint()` |
| Empezar o reiniciar una partida | `stats.reset_stats()`, `initialize_dynamic_settings()`, vaciar grupos, `_create_fleet()` y `center_ship()` |
| Ocultar el cursor | `pygame.mouse.set_visible(False)` |
| Que el juego se acelere | `settings.increase_speed()` al destruir la flota |
| Que algo no se reinicie entre partidas | definirlo en `__init__()` de `GameStats` (como `high_score`) |
| Que algo se reinicie en cada partida | definirlo en `reset_stats()` |
| Mostrar un número con comas | `f"{número:,}"` |
| Redondear a decenas | `round(número, -1)` |
| Sumar puntos por todos los aliens golpeados | `for aliens in collisions.values():` + `len(aliens)` |
| Mostrar las naves restantes con imágenes | `Group` de `Ship` (que hereda de `Sprite`) |

| | `button.py` | `scoreboard.py` | `game_stats.py` | `settings.py` |
|---|---|---|---|---|
| Clase | `Button` | `Scoreboard` | `GameStats` | `Settings` |
| Responsabilidad | el botón Play | puntaje, récord, nivel y naves | datos de la partida | valores estáticos y dinámicos |
| Cambios principales del capítulo | archivo nuevo | archivo nuevo | `score`, `high_score`, `level` | `speedup_scale`, `score_scale`, `alien_points`, `initialize_dynamic_settings()`, `increase_speed()` |

| Llamada en `_check_play_button()` | Para qué |
|---|---|
| `self.settings.initialize_dynamic_settings()` | volver las velocidades y los puntos a sus valores iniciales |
| `self.stats.reset_stats()` | naves, puntaje y nivel |
| `self.sb.prep_score()` / `prep_level()` / `prep_ships()` | actualizar las imágenes del marcador |
| `self.bullets.empty()` / `self.aliens.empty()` | quitar lo que quedaba de la partida anterior |
| `self._create_fleet()` / `self.ship.center_ship()` | flota nueva y nave centrada |
| `pygame.mouse.set_visible(False)` | ocultar el cursor |